# Imports

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import wandb
import copy
from tabulate import tabulate
import tqdm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

from diffi.utils import *

Setting up W&B

In [2]:
wandb.login()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: sanson-sebastiano-00 (sanson-sebastiano-00-universita-di-padova) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Setting a seed value for reproducibility

In [3]:
np.random.seed(0)

# Synthetic dataset

In [4]:
n_features = 20   # Total number of features
n_meaningful = 2  # Number of meaningful features

# Parameters
n_regular = 900  # Number of regular data points
n_anomalous = 100  # Number of anomalous data points

In [5]:
def generate_syn_dataset():
    """
    Generate a synthetic dataset with meaningful features in polar coordinates
    and noise features in Cartesian coordinates.
    
    The dataset consists of two types of data points:
    - Regular data points with meaningful features in polar coordinates.
    - Anomalous data points with larger radial distances, also in polar coordinates.
    
    Returns: 
        data (np.ndarray): Combined dataset with meaningful and noise features.
        labels (np.ndarray): Labels indicating regular (0) and anomalous (1) data points.
        contamination (float): Proportion of anomalous data points in the dataset.
    """
    # Generate regular data
    rho_regular = np.random.uniform(0, 3, n_regular)
    theta_regular = np.random.uniform(0, 2 * np.pi, n_regular)
    meaningful_regular = np.column_stack((rho_regular * np.cos(theta_regular), 
                                        rho_regular * np.sin(theta_regular)))

    # Generate anomalous data
    rho_anomalous = np.random.uniform(4, 30, n_anomalous)
    theta_anomalous = np.random.uniform(0, 2 * np.pi, n_anomalous)
    meaningful_anomalous = np.column_stack((rho_anomalous * np.cos(theta_anomalous), 
                                            rho_anomalous * np.sin(theta_anomalous)))

    # Combine meaningful features
    meaningful_features = np.vstack((meaningful_regular, meaningful_anomalous))

    # Generate noise features
    noise_features = np.random.normal(0, 1, (n_regular + n_anomalous, n_features - n_meaningful))

    # Combine meaningful and noise features
    X = np.hstack((meaningful_features, noise_features))

    # Create labels (0 for regular, 1 for anomalous)
    y = np.array([0] * n_regular + [1] * n_anomalous)

    contamination = n_anomalous / (n_regular + n_anomalous)

    return X, y, contamination

# Helper functions

## Logging 

In [6]:
def log_feature_importance(feature_importances, threshold_type, og_model: bool):
    """
    Log feature importance plot to Weights & Biases.

    Args:
        feature_rank (list): List of feature indices sorted by importance.
        feature_importance (list): List of feature importance values.
        og_model (bool): Whether the model is the original or a pruned version.
    """

    sorted_indices = np.argsort(feature_importances)[::-1]
    fi_std = np.std(feature_importances)

    plt.figure(figsize=(10, 5))
    plt.grid(True, axis='y', linestyle='--', alpha=0.7, zorder=0)
    plt.bar(range(len(sorted_indices)), feature_importances[sorted_indices], 
            yerr=fi_std, zorder=3)
    plt.xticks(range(len(sorted_indices)), sorted_indices)
    plt.xlabel('Feature Index')
    plt.ylabel('Feature Importance')
    plt.ylim(bottom=0)
    if og_model:
        plt.title('Feature Importance - Original Model')
        wandb.log({f"threshold_type_{threshold_type}/feature_importance_original": wandb.Image(plt)})
    else:
        plt.title('Feature Importance - Pruned Model')
        wandb.log({f"threshold_type_{threshold_type}/feature_importance_pruned": wandb.Image(plt)})
    plt.close()

In [7]:
def log_fi_heatmap(fis_in, fis_out, threshold_type):
    """
    Log feature importance heatmaps to wandb with inliers and outliers side-by-side.
    
    Args:
        fis_in (list): List of feature importance matrices for inliers, one per forest.
        fis_out (list): List of feature importance matrices for outliers, one per forest.
    """
    # Ensure we have the same number of forests in both inputs
    num_forests = len(fis_in)
    assert len(fis_out) == num_forests, "Number of forests doesn't match between inliers and outliers"
    
    for i in range(num_forests):
        # Create a figure with two subplots side by side
        fig, axes = plt.subplots(1, 2, figsize=(20, 8))
        
        # Plot inliers heatmap on the left
        sns.heatmap(fis_in[i], cmap='viridis', cbar=True, vmin=0, vmax=0.16, ax=axes[0])
        axes[0].set_title(f'Feature Importance - Inliers (Forest {i+1})', fontsize=14)
        axes[0].set_xlabel('Feature Index')
        axes[0].set_ylabel('Tree Index')
        
        # Plot outliers heatmap on the right
        sns.heatmap(fis_out[i], cmap='viridis', cbar=True, vmin=0, vmax=0.16, ax=axes[1])
        axes[1].set_title(f'Feature Importance - Outliers (Forest {i+1})', fontsize=14)
        axes[1].set_xlabel('Feature Index')
        axes[1].set_ylabel('Tree Index')
        
        plt.tight_layout()
        
        # Log the combined figure to wandb
        wandb.log({f"threshold_type_{threshold_type}/feature_importance_heatmap_forest_{i+1}": wandb.Image(fig)})
        
        # Close the figure to free memory
        plt.close(fig)

In [8]:
def log_fi_diff(og_fi, pruned_fi, threshold_type):
    """
    Log the difference in feature importance between the original and pruned model.
    
    Args:
        og_fi (np.ndarray): Feature importance of the original model.
        pruned_fi (np.ndarray): Feature importance of the pruned model.
    """
    sorted_indices = np.argsort(og_fi)[::-1]
    fi_diff = pruned_fi - og_fi

    plt.figure(figsize=(10, 5))
    plt.grid(True, axis='y', linestyle='--', alpha=0.7, zorder=0)
    plt.bar(range(len(sorted_indices)), fi_diff[sorted_indices], 
            yerr=np.std(fi_diff), zorder=3)
    plt.xticks(range(len(sorted_indices)), sorted_indices)
    plt.xlabel('Feature Index')
    plt.ylabel('Feature Importance Difference')
    plt.title('Feature Importance Difference - Original vs Pruned Model')
    wandb.log({f"threshold_type_{threshold_type}/feature_importance_difference": wandb.Image(plt)})  
    plt.close()

In [9]:
def log_estimators_summaries(iforests, og_num_trees, threshold_type, seed):
    """
    Log the number of estimators in each Isolation Forest to wandb.

    Args:
        iforests (list): List of Isolation Forest models.
        seed (int): Random seed used for the models.
    """

    # # Log per-forest statistics
    # for i, iforest in enumerate(iforests):
    #     forest_count = len(iforest.estimators_)
    #     forest_index = i + 1  # Use 1-based indexing for readability
        
    #     wandb.log({
    #         f"threshold_type_{threshold_type}/seed_{seed}/forest_{forest_index}/num_estimators": forest_count,
    #         # Percentage of trees retained compared to original 
    #         f"threshold_type_{threshold_type}/seed_{seed}/forest_{forest_index}/retention_rate": forest_count / og_num_trees,
    #         f"threshold_type_{threshold_type}/seed_{seed}/forest_{forest_index}/mean_estimators": np.mean(forest_count),
    #         f"threshold_type_{threshold_type}/seed_{seed}/forest_{forest_index}/min_estimators": np.min(forest_count),
    #         f"threshold_type_{threshold_type}/seed_{seed}/forest_{forest_index}/max_estimators": np.max(forest_count),
    #     })

    # wandb.log({
    #     f"threshold_type_{threshold_type}/seed_{seed}/overall_total_estimators": np.sum([len(iforest.estimators_) for iforest in iforests]),
    # })

    # Log summary table
    data = [[i+1, len(iforests[i].estimators_)] for i in range(len(iforests))]
    columns = ["Forest Index", "Number of Estimators"]
    wandb.log({f"threshold_type_{threshold_type}/seed_{seed}/estimators_table": wandb.Table(data=data, columns=columns)})

## Usage feature counter

In [10]:
def get_feature_usage(used_features, unique_features):
    """
    Calculating the feature usage in each tree for each forest.

    Args: 
        used_features (list): 
            - each element represents a forest and it is a list
                - each element of a forest is a tree and contains a list of features used in that tree
        unique_features (list): list of unique features used in the dataset.
    Returns:
        usage (np.ndarray): shape (num_forests, num_trees, num_features) 
            Each element is the count of how many times a feature is used in a tree of a forest.
        normalized_usage (np.ndarray): shape (num_forests, num_trees, num_features)
            Each element is the normalized count of how many times a feature is used in a tree of a forest. Normalization
            wrt to the total splits used in the tree. 
    """

    num_forests = len(used_features)
    num_trees = len(used_features[0])
    num_features = len(unique_features)

    usage = np.zeros((num_forests, num_trees, num_features), dtype=float)
    normalized_usage = np.zeros((num_forests, num_trees, num_features), dtype=float)

    # Iterate over each forest
    for f in range(num_forests):
        # Iterate over each tree in the forest
        for t in range(num_trees):
            internal_nodes_counter = 0
            # Iterate over each feature used in the tree
            for feature in used_features[f][t]:
                if feature in unique_features:
                    feature_index = unique_features.index(feature)
                    usage[f, t, feature_index] += 1
                if feature != -2:  # not a leaf node
                    internal_nodes_counter += 1
            
            # Normalize the usage by the number of internal nodes in the tree
            if internal_nodes_counter > 0:
                normalized_usage[f, t] = usage[f, t] / internal_nodes_counter
            else:
                raise ValueError(f"Tree {t} in forest {f} has no internal nodes, cannot normalize usage.")

    return usage, normalized_usage

## Meaningful feature selection methods

In [11]:
def elbow_features_selection(feature_importances):
    """
    Selecting most important features based on the elbow method.

    Args:
        feature_importances (np.ndarray): Array of shape (num_forests, num_features)
            containing the feature importances for each forest.

    Returns:
        selected_features (list): List of lists, where each inner list contains the indices of the selected features for each forest.
        threshold_gaps (np.ndarray): Array of shape (num_forests,) containing the maximum gap thresholds for each forest.
        gaps (list): List of lists, where each inner list contains the gaps between consecutive feature importances for each forest.
    """

    n_forests, n_features = feature_importances.shape

    # Get the indexes of the features sorted by importance
    sorted_indices = np.argsort(feature_importances, axis=1)[:, ::-1]

    # Sort the feature importances based on the sorted indices
    sorted_importances = np.take_along_axis(feature_importances, sorted_indices, axis=1)

    selected_features, gaps = [], []
    threshold_gaps = np.zeros(n_forests, dtype=float)

    if n_features < 2: # Not enough features to compute gaps
        for f in range(n_forests):
            selected_idx = sorted_indices[f, :].tolist()
            selected_features.append(selected_idx)
            gaps.append([])
    else:
        # Calculate gaps between consecutive feature importances
        gaps_batch = sorted_importances[:, :-1] - sorted_importances[:, 1:]

        # Get the maximum gap for each forest
        max_gap_indices = np.argmax(gaps_batch + 1e-12 * np.random.rand(*gaps_batch.shape), axis=1)
        threshold_gaps = np.take_along_axis(gaps_batch, max_gap_indices[:, np.newaxis], axis=1).squeeze()

        # Select features based on the maximum gap
        for f in range(n_forests):
            num_to_select = max_gap_indices[f] + 1
            selected_idx_for_forest = sorted_indices[f, :num_to_select].tolist()
            selected_features.append(selected_idx_for_forest)
            gaps.append(gaps_batch[f, :].tolist())

    return selected_features, threshold_gaps, gaps

In [12]:
# TODO: add soft threshold

## Selection of trees to be removed

In [13]:
def get_unique_usage_per_tree(normalized_usages, feature_importances, selected_features):
    """
    Compute the unique usage of selected features for each tree in each forest as weighted sum of features usage weighted by their importance.

    Args:
        normalized_usages (np.ndarray): Array of shape (num_forests, num_trees, num_features)
            containing the normalized feature usage for each tree in each forest.
        feature_importances (np.ndarray): Array of shape (num_forests, num_features)
            containing the feature importances for each forest.
        selected_features (list): List of lists, where each inner list contains the indices of the selected features for each forest.

    Returns:
        unique_usages (np.ndarray): Array of shape (num_forests, num_trees) containing the weighted usage for each tree in each forest.
    """

    n_forests, n_trees, n_features = normalized_usages.shape
    unique_usages = np.zeros((n_forests, n_trees), dtype=float)

    for f in range(n_forests):
        forest_selected_features = set(selected_features[f])
        forest_total_importance = np.sum(feature_importances[f, :])
        assert forest_total_importance > 0, f"Forest {f} has zero total importance, cannot compute unique usage."

        for t in range(n_trees):
            current_tree_weighted_usage = 0.0
            current_tree_weight_sum = 0.0

            for feature in range(n_features):
                if feature in forest_selected_features:
                    current_tree_weighted_usage += normalized_usages[f, t, feature] * feature_importances[f, feature]
                    current_tree_weight_sum += feature_importances[f, feature]

            assert current_tree_weight_sum > 0, f"Tree {t} in forest {f} has zero total importance, cannot compute unique usage."
            unique_usages[f, t] = current_tree_weighted_usage / current_tree_weight_sum
            
    return unique_usages

In [14]:
def get_unique_avg_usage_per_forest(normalized_usages, selected_features):
    """
    Compute the average usage of selected features across all trees in each forest.

    Args:
        normalized_usages (np.ndarray): Array of shape (num_forests, num_trees, num_features)
            containing the normalized feature usage for each tree in each forest.
        selected_features (list): List of lists, where each inner list contains the indices of the selected features for each forest.

    Returns:
        unique_avg_usage (np.ndarray): Array of shape (num_forests, num_features) containing the average usage of selected feature for each forest.
    """

    n_forests, _, n_features = normalized_usages.shape
    unique_avg_usage = np.zeros((n_forests, n_features), dtype=float)

    for f in range(n_forests):
        forest_selected_features = set(selected_features[f])

        for feature in range(n_features):
            if feature in forest_selected_features:
                unique_avg_usage[f, feature] = np.mean(normalized_usages[f, :, feature])

    return unique_avg_usage

def majority_voting(normalized_usages, selected_features, feature_importances):
    """ 
    Apply majority voting to compute weighted voting score 
    Args:
        normalized_usages (np.ndarray): Array of shape (num_forests, num_trees, num_features)
            containing the normalized feature usage for each tree in each forest.
        selected_features (list): List of lists, where each inner list contains the indices of the selected features for each forest.
        feature_importances (np.ndarray): Array of shape (num_forests, num_features)
            containing the feature importances for each forest.
    Returns:
        voting_scores (np.ndarray): Array of shape (num_forests, num_trees) containing the voting score for each tree in each forest.
    """
    
    n_forests, n_trees, n_features = normalized_usages.shape
    binary_scores = np.zeros((n_forests, n_trees, n_features), dtype=int)
    voting_scores = np.zeros((n_forests, n_trees), dtype=float)

    unique_avg_usage = get_unique_avg_usage_per_forest(normalized_usages, selected_features)

    for f in range(n_forests):
        forest_selected_features = set(selected_features[f])
        forest_total_importance = np.sum(feature_importances[f, :])
        assert forest_total_importance > 0, f"Forest {f} has zero total importance, cannot compute voting score."

        for t in range(n_trees):
            current_tree_weight_sum = 0.0

            for feature in range(n_features):
                if feature in forest_selected_features:
                    if normalized_usages[f, t, feature] > unique_avg_usage[f, feature]:
                        binary_scores[f, t, feature] = 1
                    current_tree_weight_sum += feature_importances[f, feature]

            assert current_tree_weight_sum > 0, f"Tree {t} in forest {f} has zero total importance, cannot compute voting score."
            voting_scores[f, t] = np.sum(np.multiply(binary_scores[f, t, :], feature_importances[f, :])) 

    return voting_scores

In [15]:
def remove_trees(iforests, unique_usages, thresholds):
    """
    For each forest, remove trees that have a feature usage below a certain threshold.

    Args:
        iforests (list): List of forests, each containing a list of trees.
        unique_usages (np.ndarray): Array of shape (num_forests, num_trees, num_features) 
            containing the usage of each feature in each tree.
        thresholds (list): List of thresholds for each forest to determine which trees to remove.

    Returns:
        pruned_forests (list): List of pruned forests with trees removed based on the thresholds.
    """
    pruned_forests = copy.deepcopy(iforests)

    n_forests = len(iforests)
    keep_mask = unique_usages > thresholds[:, np.newaxis]

    for f in range(n_forests):
        # Get the index of the forest to prune
        current_forest = pruned_forests[f]
        current_mask = keep_mask[f]
        og_estimators = current_forest.estimators_
        og_features = current_forest.estimators_features_

        # Filter and update the estimators and features based on the mask
        pruned_forests[f].estimators_ = [est for idx, est in enumerate(og_estimators) if current_mask[idx]]
        pruned_forests[f].estimators_features_ = [feat for idx, feat in enumerate(og_features) if current_mask[idx]]

        # Update also the internal attributes of the forest
        og_decision_paths = current_forest._decision_path_lengths
        pruned_forests[f]._decision_path_lengths = [path for idx, path in enumerate(og_decision_paths) if current_mask[idx]]

        og_avg_paths = current_forest._average_path_length_per_tree
        pruned_forests[f]._average_path_length_per_tree = [avg_path for idx, avg_path in enumerate(og_avg_paths) if current_mask[idx]]
        pruned_forests[f].n_estimators_ = len(pruned_forests[f].estimators_)
        
    return pruned_forests


In [16]:
def random_trees_removal(iforests, num_trees_to_remove):
    """
    For each forest, remove trees that have a feature usage below a certain threshold.

    Args:
        iforests (list): List of forests, each containing a list of trees.
        unique_usages (np.ndarray): Array of shape (num_forests, num_trees, num_features) 
            containing the usage of each feature in each tree.
        thresholds (list): List of thresholds for each forest to determine which trees to remove.

    Returns:
        pruned_forests (list): List of pruned forests with trees removed based on the thresholds.
    """
    pruned_forests = copy.deepcopy(iforests)

    n_forests = len(iforests)

    for f in range(n_forests):
        # Get the index of the forest to prune
        current_forest = pruned_forests[f]
        n_trees = len(current_forest.estimators_)

        if num_trees_to_remove >= n_trees:
            raise ValueError(f"Cannot remove {num_trees_to_remove} trees from forest {f} with only {n_trees} trees.")
        
        # Mask of trees to keep (True) and remove (False)
        keep_mask = np.ones(n_trees, dtype=bool)
        indices_to_remove = np.random.choice(n_trees, num_trees_to_remove, replace=False)
        keep_mask[indices_to_remove] = False

        og_estimators = current_forest.estimators_
        og_features = current_forest.estimators_features_

        # Filter and update the estimators and features based on the mask
        pruned_forests[f].estimators_ = [est for idx, est in enumerate(og_estimators) if keep_mask[idx]]
        pruned_forests[f].estimators_features_ = [feat for idx, feat in enumerate(og_features) if keep_mask[idx]]

        # Update also the internal attributes of the forest
        og_decision_paths = current_forest._decision_path_lengths
        pruned_forests[f]._decision_path_lengths = [path for idx, path in enumerate(og_decision_paths) if keep_mask[idx]]

        og_avg_paths = current_forest._average_path_length_per_tree
        pruned_forests[f]._average_path_length_per_tree = [avg_path for idx, avg_path in enumerate(og_avg_paths) if keep_mask[idx]]
        pruned_forests[f].n_estimators_ = len(pruned_forests[f].estimators_)
        
    return pruned_forests


# Experiments

## Parameters

In [17]:
num_trees = 100
max_samples = 256  
num_forests = 10
test_size = 0   # 0.1
seeds = [0, 1, 2, 3, 4]

In [18]:
threshold_type = 'percentile'  # 'random', 'mean', 'majority_voting', 'percentile' 
# selection_method = 'elbow'  # 'elbow', 'soft_threshold'
percentile = 80             # for percentile thresholding
num_trees_to_remove = 80    # for random removal

## Experiment loop

In [19]:
# Initialize results list to store the results of each experiment
results = []
# Generate synthetic dataset
X, y, contamination = generate_syn_dataset()

f1_over_seeds = np.zeros((len(seeds), num_forests))
avg_precision_over_seeds = np.zeros((len(seeds), num_forests))

# Get the unique features used in the dataset
all_features = range(X.shape[1])

for seed in tqdm.tqdm(seeds, desc="Running experiments", total=len(seeds)):

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed) if test_size > 0 else (X, X, y, y)
    X_train, y_train = shuffle(X_train, y_train, random_state=seed)

    # Init Weights & Biases run
    run = wandb.init(
        project="if_optimization",
        name=f"diffi_syn_seed_{seed}_forest_{num_forests}_trees_{num_trees}",
        config={
            "seed": seed,
            "dataset": "synthetic",
            "num_forests": num_forests,
            "num_trees": num_trees,
            "contamination": contamination,
            "max_samples": max_samples,
            "test_size": test_size,
            "thresholds": threshold_type,
            "percentile_value": percentile if threshold_type == 'percentile' else "N/A",            # for percentile thresholding
            "num_trees_to_remove": num_trees_to_remove if threshold_type == 'random' else "N/A",    # for random removal

        },
    )

    f1s, avg_precisions, fi_og, iforests, used_features, fis_out, fis_in = diffi_ranks(
        X_train,
        X_test,
        y_test,
        seed,
        n_iters=num_forests,
        contamination=contamination,
        num_trees=num_trees,  
    )

    tmp = random_trees_removal(iforests, num_trees_to_remove=10)

    mean_f1 = np.mean(f1s)
    mean_avg_precision = np.mean(avg_precisions)
    mean_fi_og = np.mean(fi_og, axis=0)

    log_feature_importance(mean_fi_og, threshold_type, og_model=True)
    log_fi_heatmap(fis_in, fis_out, threshold_type)

    # ???
    f1_over_seeds[seed, :] = f1s
    avg_precision_over_seeds[seed, :] = avg_precisions

    pruned_iforests =  []
    if threshold_type == 'random':
        pruned_iforests = random_trees_removal(iforests, num_trees_to_remove=num_trees_to_remove)
    else:   # threshold_type in ['mean', 'percentile', 'majority_voting']
        # feature selection
        selected_features, gap_thresholds, gaps = elbow_features_selection(fi_og)
        
        # extract only the feature importances of the selected features
        unique_fi_og = np.zeros((num_forests, fi_og.shape[1]), dtype=float)
        for f in range(num_forests):
            forest_selected_features = set(selected_features[f])
            
            for sf in forest_selected_features:
                unique_fi_og[f, sf] = fi_og[f, sf]
        # normalize the unique feature importances per forest
        normalized_unique_fi_og = np.divide(unique_fi_og, np.sum(unique_fi_og, axis=1, keepdims=True))
        # get feature usage
        usages, normalized_usages = get_feature_usage(used_features, all_features)

        thresholds = np.zeros(num_forests, dtype=float)

        if threshold_type == 'mean':
            unique_usages = get_unique_usage_per_tree(normalized_usages, normalized_unique_fi_og, selected_features)
            for f in range(num_forests):
                thresholds[f] = np.mean(unique_usages[f, :])
            pruned_iforests = remove_trees(iforests, unique_usages, thresholds)
        elif threshold_type == 'percentile':
            unique_usages = get_unique_usage_per_tree(normalized_usages, normalized_unique_fi_og, selected_features)
            for f in range(num_forests):
                thresholds[f] = np.percentile(unique_usages[f, :], percentile)
            pruned_iforests = remove_trees(iforests, unique_usages, thresholds)
        else:   # 'majority_voting'
            voting_scores = majority_voting(normalized_usages, selected_features, normalized_unique_fi_og)
            pruned_iforests = remove_trees(iforests, voting_scores, np.array([0.5]*num_forests))

    log_estimators_summaries(pruned_iforests, run.config.num_trees, threshold_type, seed)

    f1s_pruned, avg_precisions_pruned, fi_pruned, fis_out_pruned, fis_in_pruned = diffi_ranks_evaluation_only(
        X_test, 
        y_test, 
        pruned_iforests
    )
        
    mean_f1_pruned = np.mean(f1s_pruned)
    mean_avg_precision_pruned = np.mean(avg_precisions_pruned)
    mean_fi_pruned = np.mean(fi_pruned, axis=0)

    log_feature_importance(mean_fi_pruned, threshold_type, og_model=False)
    log_fi_diff(mean_fi_og, mean_fi_pruned, threshold_type)

    results.append([
        seed,
        f"{mean_f1:.4f}",
        f"{mean_f1_pruned:.4f}",
        f"{mean_avg_precision:.4f}",
        f"{mean_avg_precision_pruned:.4f}",
    ])

headers = ["Seed", "F1 Score", "F1 Score Pruned", "Avg Precision", "Avg Precision Pruned"]
table = tabulate(results, headers=headers, tablefmt="grid")

# log performance metrics to wandb
wandb.log({"results_table": wandb.Table(data=results, columns=headers)})

Running experiments:   0%|          | 0/5 [00:00<?, ?it/s]

Running experiments:  20%|██        | 1/5 [00:09<00:37,  9.46s/it]

Running experiments:  40%|████      | 2/5 [00:20<00:31, 10.48s/it]

Running experiments:  60%|██████    | 3/5 [00:31<00:21, 10.64s/it]

Running experiments:  80%|████████  | 4/5 [00:42<00:10, 10.71s/it]

Running experiments: 100%|██████████| 5/5 [00:52<00:00, 10.58s/it]


In [20]:
run.finish()